# 阶段总结：核查缓存 → 统一代码 → 短跑测时间


本节交付：已有缓存的完整性核查、七种方法共用的模型/损失/训练步、GPU 短跑耗时表。不会重新选模型、计算测试成绩或覆盖旧实验。没有预填实测耗时。

准备：将 `1013实验成果包与结论.zip` **自行解压**到 Drive，保留 `文件校验清单.json` 与 `reused_v1` 等目录。已有完整特征缓存时不需要图片或 Teacher 权重。本节核查既有缓存，不自动重建缺失/损坏缓存：缺失时修复原包或重新导出，再继续，不能混用重新生成的特征。

建议按节运行。短跑默认七种方法各 3 次，每次预热 10 步、测量 50 步。模型是临时随机初始化的 MLP，不读取或保存正式 checkpoint。


## 1．环境

沿用 Colab 自带 PyTorch；本节只需 numpy、torch、pandas，不下载 CLIP 或 Teacher。真实环境会写入报告。历史实验环境为 T4、torch 2.11.0+cu130、numpy 2.2.6；若当前环境不同，本次数据只描述当前机器的工程成本，不能冒充历史严格复现。


In [1]:
import os, sys, subprocess, importlib.util
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
missing=[name for name in ['numpy','pandas'] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
import torch, numpy as np, pandas as pd
print('torch:',torch.__version__,'numpy:',np.__version__,'CUDA:',torch.version.cuda)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无；可先核查缓存，计时需要 GPU')


torch: 2.11.0+cu130 numpy: 2.1.3 CUDA: 13.0
GPU: Tesla T4


## 2．设置路径

修改 `SOURCE`，使它指向解压后的 `10月13日_任务交付`（或其父文件夹）；也可直接指向其中的 `reused_v1`。`WORK` 必须在原成果包之外。原始文件只读。每次短跑建立新目录；中断后重新短跑，旧记录保留。


In [2]:
from pathlib import Path
if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/1013实验成果包与结论/10月13日_任务交付')
WORK = Path('/content/drive/MyDrive/CLIP_Jev_阶段缓存短跑_20261022')
assert SOURCE.is_dir(), '请修改 SOURCE，指向已解压的成果包目录。'
assert not WORK.resolve().is_relative_to(SOURCE.resolve()) and not SOURCE.resolve().is_relative_to(WORK.resolve()), '输入与输出目录须相互独立。'
WORK.mkdir(parents=True,exist_ok=True)


Mounted at /content/drive


## 3．写出统一运行代码

统一规则：图文头 2048→512→3；单模态头 512→2048→3；ReLU、dropout=0.1；FP32、AdamW、lr=0.001、weight_decay=0.0001。所有方法共用 `make_head / select_features / target / loss_fn` 与同一个训练步。

软 KD：`(1−α) CE + α τ² KL(teacher || student)`；硬 KD：`(1−α) CE(gold) + α CE(teacher argmax)`。校准先执行 `(z+[bE,0,0])/T`，再除以蒸馏温度 τ。默认 α=0.75、τ=2，属于后续实验方案；早期 v1 的 α=0.5 仍保留在原包，本节不改写历史。


In [3]:
import hashlib, importlib.util
RUNTIME_SOURCE = '"""Read-only historical cache audit and disposable, unified training benchmark."""\nfrom pathlib import Path\nimport hashlib, json, time, math, random, platform, uuid, csv\nimport numpy as np\n\nREV = \'3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268\'\nMETHODS = [\'CE\', \'KD_raw\', \'KD_calibrated\', \'KD_base\', \'KD_hard\', \'CE_image\', \'CE_text\']\n\ndef check(ok, message):\n    if not ok: raise ValueError(message)\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1048576), b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef read(path): return json.loads(Path(path).read_text(encoding=\'utf-8-sig\'))\ndef rows(path): return [json.loads(s) for s in Path(path).read_text(encoding=\'utf-8-sig\').splitlines() if s.strip()]\ndef save(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.\' + uuid.uuid4().hex + \'.partial\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\ndef locate(source):\n    source = Path(source).resolve()\n    candidates = [source, source/\'reused_v1\']\n    candidates += list(source.glob(\'**/reused_v1\'))\n    matches = sorted(set(p for p in candidates if (p/\'students/training_lock.json\').is_file()))\n    check(len(matches) == 1, \'SOURCE 必须指向已解压的单个成果包或 Student_v1 目录；未唯一找到训练锁。\')\n    return matches[0]\n\ndef audit(source):\n    old = locate(source); lock = read(old/\'students/training_lock.json\')\n    files = {}; data = {}; seen = set(); seen_samples = set()\n    def record(p):\n        digest=sha(p); files[p.relative_to(old).as_posix()] = digest; return digest\n    revision = read(old/\'features/clip_revision.json\')\n    check(revision[\'revision\'] == REV, \'CLIP revision 不符\')\n    record(old/\'features/clip_revision.json\'); record(old/\'students/training_lock.json\')\n    # Only read held-out manifests to verify isolation; never read their features/predictions.\n    for split, n in [(\'train\',6000),(\'val_fit\',400),(\'val_select\',400),(\'test\',1000)]:\n        manifest = old/f\'handoff/manifests/{split}.jsonl\'; rr = rows(manifest); record(manifest)\n        ids = [str(r[\'image_id\']) for r in rr]; sid = [r[\'sample_id\'] for r in rr]\n        check(len(rr)==n and len(set(ids))==n and len(set(sid))==n, f\'{split} 数量/重复 ID 异常\')\n        check(not seen.intersection(ids) and not seen_samples.intersection(sid), \'划分间 ID 泄漏\')\n        seen.update(ids); seen_samples.update(sid)\n        check(all(r[\'label\'] in [0,1,2] and r[\'split\']==split for r in rr), \'标签或 split 异常\')\n        if split not in [\'train\',\'val_select\']: continue\n        p = old/f\'features/{split}.npz\'; meta_path = old/f\'features/{split}_meta.json\'\n        meta = read(meta_path); record(meta_path)\n        digest = record(p)\n        check(digest == meta[\'npz_sha256\'] == lock[\'feature_hashes\'][split], f\'{split} 特征哈希不符\')\n        check(meta[\'manifest_sha256\']==sha(manifest) and meta[\'revision\']==REV, \'特征来源不符\')\n        check(meta[\'feature_order\']==[\'v\',\'t\',\'abs(v-t)\',\'v*t\'] and meta[\'frozen_encoder\'], \'特征定义不符\')\n        with np.load(p, allow_pickle=False) as z: d = {k:z[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        x = d[\'x\']; v,t = x[:,:512],x[:,512:1024]\n        check(x.shape==(n,2048) and x.dtype==np.float32 and np.isfinite(x).all(), \'特征形状/精度/有限性异常\')\n        check(d[\'ids\'].tolist()==sid and d[\'image_ids\'].astype(str).tolist()==ids and d[\'y\'].tolist()==[r[\'label\'] for r in rr], \'缓存顺序或标签不符\')\n        check(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4), \'特征未归一化\')\n        check(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6), \'特征分段不符\')\n        image_audit = old/\'handoff/manifests/image_audit.jsonl\'\n        check(record(image_audit)==meta[\'image_audit_sha256\'], \'图片审计来源变化\')\n        d[\'rows\']=rr; data[split]=d\n    runpath = old/\'handoff/predictions/train/run.json\'; run = read(runpath); record(runpath)\n    check(run[\'saved_order\']==[\'E\',\'N\',\'C\'] and run[\'manifest_sha256\']==files[\'handoff/manifests/train.jsonl\'], \'Teacher 顺序/来源不符\')\n    logits = {}\n    image_rows = rows(old/\'handoff/manifests/image_audit.jsonl\')\n    image_map = {r[\'sample_id\']:r for r in image_rows}\n    check(len(image_map)==len(image_rows), \'图片审计重复 ID\')\n    for method in [\'jev\',\'base\']:\n        p = old/f\'handoff/predictions/train/{method}.jsonl\'; rr=rows(p); digest=record(p)\n        if method==\'jev\': check(digest==lock[\'teacher_train_sha256\'], \'Jev 缓存与训练锁不符\')\n        byid={r[\'sample_id\']:r for r in rr}\n        check(len(byid)==len(rr)==6000 and set(byid)==set(data[\'train\'][\'ids\']), \'教师缓存缺失/重复/多余样本\')\n        ordered=[byid[r[\'sample_id\']] for r in data[\'train\'][\'rows\']]\n        for r, pred in zip(data[\'train\'][\'rows\'],ordered):\n            check(pred[\'model\']==method and all(pred[k]==r[k] for k in [\'label\',\'image_id\',\'image_file\',\'hypothesis\',\'split\']), \'教师缓存内容错配\')\n            check(pred[\'image_sha256\']==image_map[r[\'sample_id\']][\'file_sha256\'], \'教师图片哈希错配\')\n        z=np.asarray([r[\'logits\'] for r in ordered],dtype=np.float64)\n        check(z.shape==(6000,3) and np.isfinite(z).all(), \'教师 logits 异常\'); logits[method]=z\n    calpath=old/\'handoff/results/calibration.json\'; cal=read(calpath)\n    check(record(calpath)==lock[\'calibration_sha256\'] and cal[\'labels\']==[\'E\',\'N\',\'C\'], \'校准文件不符\')\n    params=cal[\'models\'][\'jev\'][\'selected_params\']\n    check(np.isfinite([params[\'T\'],params[\'b\']]).all() and params[\'T\']>0, \'校准参数异常\')\n    package_manifest=old.parent/\'文件校验清单.json\'\n    if package_manifest.exists():\n        hashes=read(package_manifest)\n        for name,digest in files.items(): check(hashes[\'reused_v1/\'+name]==digest, \'成果包总清单不符：\'+name)\n    else:\n        raise ValueError(\'需要10月13日成果包的文件校验清单.json，以验证 base 缓存来源。请指定完整成果包。\')\n    report=dict(status=\'PASS\', source=str(old), input_sha256=files, clip_revision=REV,\n                counts={s:len(d[\'y\']) for s,d in data.items()}, label_order=[\'E\',\'N\',\'C\'],\n                scope=\'历史缓存完整性与来源；未重新读取原图或重新推理 CLIP/Teacher\', calibration=params)\n    return dict(old=old, data=data, logits=logits, params=params, report=report)\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)\n\ndef make_head(method):\n    import torch.nn as nn\n    d,h=(512,2048) if method in [\'CE_image\',\'CE_text\'] else (2048,512)\n    return nn.Sequential(nn.Linear(d,h),nn.ReLU(),nn.Dropout(.1),nn.Linear(h,3))\n\ndef select_features(x, method):\n    return x[:,:512] if method==\'CE_image\' else x[:,512:1024] if method==\'CE_text\' else x\n\ndef target(ctx, method, tau):\n    if method in [\'CE\',\'CE_image\',\'CE_text\']: return None\n    z=ctx[\'logits\'][\'base\' if method==\'KD_base\' else \'jev\'].copy()\n    if method==\'KD_hard\': return z.argmax(1)\n    if method==\'KD_calibrated\': z=(z+np.array([ctx[\'params\'][\'b\'],0,0]))/ctx[\'params\'][\'T\']\n    z=z/tau; z-=z.max(1,keepdims=True); p=np.exp(z); return (p/p.sum(1,keepdims=True)).astype(np.float32)\n\ndef loss_fn(z,y,q,method,alpha,tau):\n    import torch.nn.functional as F\n    ce=F.cross_entropy(z,y)\n    if q is None: return ce\n    kd=F.cross_entropy(z,q) if method==\'KD_hard\' else tau*tau*F.kl_div(F.log_softmax(z/tau,dim=-1),q,reduction=\'batchmean\')\n    return (1-alpha)*ce+alpha*kd\n\ndef benchmark(ctx, out, methods=METHODS, seed=20261010, alpha=.75, tau=2., batch=128, warmup=10, steps=50, repeats=3):\n    import torch\n    check(torch.cuda.is_available(), \'短跑要求 GPU；不自动用 CPU 替代计时。\')\n    check(0<=alpha<=1 and tau>0 and batch>0 and warmup>=1 and steps>=1 and repeats>=1, \'计时参数不合法\')\n    check(len(methods)>0 and len(set(methods))==len(methods) and set(methods)<=set(METHODS), \'未知或重复方法\')\n    out=Path(out); out.mkdir(parents=True, exist_ok=False)\n    torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=True\n    env=dict(torch=str(torch.__version__), numpy=np.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),\n             python=platform.python_version(), cudnn=torch.backends.cudnn.version(), dtype=\'float32\', tf32_matmul=False, tf32_cudnn=True)\n    config=dict(methods=list(methods), seed=seed, alpha=alpha,tau=tau,batch=batch,warmup=warmup,steps=steps,repeats=repeats,\n                scope=\'GPU-resident cached features -> head training step; forward/loss/backward/AdamW; synchronized per step\',\n                excludes=[\'CLIP/Teacher inference\',\'disk read\',\'host-to-device copy\',\'validation\',\'checkpoint I/O\'], runtime_sha256=sha(__file__))\n    save(out/\'protocol.json\',dict(config=config,environment=env,cache_audit=ctx[\'report\']))\n    x=torch.tensor(ctx[\'data\'][\'train\'][\'x\'],device=\'cuda\'); y=torch.tensor(ctx[\'data\'][\'train\'][\'y\'],device=\'cuda\',dtype=torch.long)\n    check(batch<=len(y), \'batch 超过样本数\')\n    # Exact same sequence of full batches for every method and repeat; no tail batch in step timing.\n    g=torch.Generator(device=\'cpu\').manual_seed(seed)\n    batches=[]\n    while len(batches)<warmup+steps:\n        batches.extend(ix for ix in torch.randperm(len(y),generator=g).split(batch) if len(ix)==batch)\n    batches=[ix.to(\'cuda\') for ix in batches[:warmup+steps]]\n    raw=[]; summary=[]\n    for method in methods:\n        qn=target(ctx,method,tau); q=None if qn is None else torch.tensor(qn,device=\'cuda\',dtype=torch.long if method==\'KD_hard\' else torch.float32)\n        xx=select_features(x,method); times=[]; peaks=[]\n        for repeat in range(repeats):\n            seed_all(seed); model=make_head(method).cuda().train(); opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.0001)\n            def step(ix):\n                opt.zero_grad(set_to_none=True)\n                loss=loss_fn(model(xx[ix]),y[ix],None if q is None else q[ix],method,alpha,tau)\n                loss.backward(); opt.step(); return loss\n            for ix in batches[:warmup]: step(ix)\n            torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()\n            for j,ix in enumerate(batches[warmup:]):\n                torch.cuda.synchronize(); start=time.perf_counter(); loss=step(ix); torch.cuda.synchronize()\n                ms=(time.perf_counter()-start)*1000\n                value=float(loss.detach().cpu()); check(math.isfinite(value), \'短跑损失非有限\')\n                times.append(ms); raw.append(dict(method=method,repeat=repeat+1,step=j+1,ms=ms,loss=value))\n            peaks.append(torch.cuda.max_memory_allocated()/1024**2)\n            del opt,model; torch.cuda.empty_cache()\n        mean=float(np.mean(times)); median=float(np.median(times))\n        summary.append(dict(method=method,mean_step_ms=mean,median_step_ms=median,p95_step_ms=float(np.percentile(times,95)),\n                            samples_per_second=batch*1000/mean,peak_allocated_MiB=max(peaks),\n                            estimated_30_epochs_3_seeds_train_only_seconds=mean/1000*math.ceil(len(y)/batch)*30*3))\n        save(out/\'partial_results.json\',dict(summary=summary,raw=raw)); print(method, summary[-1],flush=True)\n        del q\n    save(out/\'results.json\',dict(summary=summary,raw=raw,note=\'工程短跑；全预算仅线性粗估，末批按整批估算；不含验证/保存，不能作为正式训练耗时或端到端加速比。\'))\n    with (out/\'timing_summary.csv\').open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(summary[0])); w.writeheader(); w.writerows(summary)\n    return summary\n'
assert hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest() == 'fc464669c08fd73dfd0449a92ba3b37e28595debc64aa32f0b11eac559cebd65'
RUNTIME = WORK/'stage_runtime.py'
RUNTIME.write_text(RUNTIME_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('stage_runtime',RUNTIME)
stage=importlib.util.module_from_spec(spec)
spec.loader.exec_module(stage)
print('统一代码已写入：',RUNTIME)


统一代码已写入： /content/drive/MyDrive/CLIP_Jev_阶段缓存短跑_20261022/stage_runtime.py


## 4．核查缓存并锁定输入

检查 train=6000、val_select=400 的特征、教师 train logits、校准参数和原训练锁；读取四个划分的清单以核查图像隔离，但不读取测试特征或预测、不评分。检查 E/N/C 顺序、归一化及 `[v,t,|v−t|,v*t]`、来源哈希、重复/缺失 ID、标签、假设与图片哈希。

PASS 只证明缓存与保存的来源记录一致，不表示本次重新核验了原始图片、CLIP 数值输出或包外数据使用史。任一不一致会中止，不会静默忽略或修补。


In [4]:
from datetime import datetime, timezone, timedelta
import uuid
AUDIT_DIR = WORK/('audit_'+datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
AUDIT_DIR.mkdir()
try:
    ctx=stage.audit(SOURCE)
except Exception as exc:
    stage.save(AUDIT_DIR/'cache_audit_FAILED.json',{'status':'FAIL','error':str(exc)})
    raise
stage.save(AUDIT_DIR/'cache_audit.json',ctx['report'])
print('PASS：缓存可复用；未运行模型。',ctx['report']['counts'])
print('报告：',AUDIT_DIR/'cache_audit.json')


PASS：缓存可复用；未运行模型。 {'train': 6000, 'val_select': 400}
报告： /content/drive/MyDrive/CLIP_Jev_阶段缓存短跑_20261022/audit_20261023_154824_0d5819/cache_audit.json


## 5．短跑配置

默认覆盖 CE、原始/校准/base 软 KD、硬 KD、仅图像 CE、仅文本 CE。所有方法使用相同初始化种子与批次顺序；每次重复重新初始化模型和优化器。输入常驻 GPU；预热不计入结果。每一步前后 CUDA 同步，避免只测 CPU 提交时间。

计时包括索引、前向、损失、反向与更新，不包括磁盘读取、传入 GPU、CLIP/Teacher 推理、验证或存盘。逐步同步有额外开销，因此这是统一条件下的工程短跑，不是正式训练总时长或部署端到端时延。


In [5]:
METHODS = ['CE','KD_raw','KD_calibrated','KD_base','KD_hard','CE_image','CE_text']
SEED = 20261010
ALPHA, TAU = 0.75, 2.0
BATCH_SIZE = 128
WARMUP_STEPS, MEASURE_STEPS, REPEATS = 10, 50, 3
print('总训练步数：',len(METHODS)*REPEATS*(WARMUP_STEPS+MEASURE_STEPS))


总训练步数： 1260


## 6．运行短跑

只运行本格才会创建临时模型并训练短跑。若显存不足，先关闭其他 GPU 工作；更改 batch 后重新建立本次报告，不与原 batch 的结果直接混合。峰值显存为 PyTorch allocated，包含常驻输入/目标/模型等，不等于整卡占用。


In [6]:
# 再检查一次输入，避免核查后文件被更换。
fresh=stage.audit(SOURCE)
assert fresh['report']==ctx['report'], '缓存核查后输入发生变化。'
del fresh
RUN = WORK/('short_run_'+datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
summary=stage.benchmark(ctx,RUN,methods=METHODS,seed=SEED,alpha=ALPHA,tau=TAU,
                        batch=BATCH_SIZE,warmup=WARMUP_STEPS,steps=MEASURE_STEPS,repeats=REPEATS)
display(pd.DataFrame(summary))
print('结果目录：',RUN)


CE {'method': 'CE', 'mean_step_ms': 2.1344911933332846, 'median_step_ms': 2.1409899999866866, 'p95_step_ms': 2.313503299970421, 'samples_per_second': 59967.45285236404, 'peak_allocated_MiB': 140.03857421875, 'estimated_30_epochs_3_seeds_train_only_seconds': 9.028897747799796}
KD_raw {'method': 'KD_raw', 'mean_step_ms': 2.5558064533311153, 'median_step_ms': 2.5314669999829675, 'p95_step_ms': 2.9495778499892804, 'samples_per_second': 50082.03959778369, 'peak_allocated_MiB': 140.107421875, 'estimated_30_epochs_3_seeds_train_only_seconds': 10.811061297590618}
KD_calibrated {'method': 'KD_calibrated', 'mean_step_ms': 2.592949213333592, 'median_step_ms': 2.5139434999914556, 'p95_step_ms': 3.3255467000088834, 'samples_per_second': 49364.638282073574, 'peak_allocated_MiB': 140.107421875, 'estimated_30_epochs_3_seeds_train_only_seconds': 10.968175172401093}
KD_base {'method': 'KD_base', 'mean_step_ms': 2.601908186668046, 'median_step_ms': 2.57621450001011, 'p95_step_ms': 2.9415461999690242, 'sa

,method,mean_step_ms,median_step_ms,p95_step_ms,samples_per_second,peak_allocated_MiB,estimated_30_epochs_3_seeds_train_only_seconds
0,CE,2.134491,2.140990,2.313503,59967.452852,140.038574,9.028898
1,KD_raw,2.555806,2.531467,2.949578,50082.039598,140.107422,10.811061
2,KD_calibrated,2.592949,2.513943,3.325547,49364.638282,140.107422,10.968175
3,KD_base,2.601908,2.576215,2.941546,49194.664384,140.107422,11.006072
4,KD_hard,2.555240,2.450971,3.387719,50093.149777,140.084473,10.808664
5,CE_image,2.211292,2.132641,2.979901,57884.700051,140.202637,9.353767
6,CE_text,2.164720,2.141172,2.408326,59130.038408,140.202637,9.156767


结果目录： /content/drive/MyDrive/CLIP_Jev_阶段缓存短跑_20261022/short_run_20261023_155033_088486


## 7．阶段总结与导出

`results.json` 保留每一步耗时和损失；`timing_summary.csv` 提供均值、中位数、P95、吞吐、显存及 30 epoch×3 seed 的训练步粗估。粗估按每轮 ceil(6000/batch) 步线性换算，末批按整批估计，**不含验证、保存、加载等成本**。短跑损失仅检查有限性，不用于判断模型效果。

完成标准：缓存 PASS，七种方法全部成功，协议/环境/原始时间与汇总文件齐全。得到这些输出后，本节结束；不继续正式训练、模型选择或最终测试。


In [7]:
import shutil
result=stage.read(RUN/'results.json')
assert len(result['summary'])==len(METHODS)
assert len(result['raw'])==len(METHODS)*REPEATS*MEASURE_STEPS
note = ('阶段核查已完成。缓存：PASS。统一训练入口：'+', '.join(METHODS)+
        '。已完成 GPU 短跑，具体耗时见 timing_summary.csv。'+
        '本次只核查既有缓存并测量临时分类头训练步，未新增正式模型或测试成绩。')
(RUN/'阶段总结.txt').write_text(note,encoding='utf-8')
shutil.copy2(RUNTIME,RUN/'stage_runtime.py')
archive=shutil.make_archive(str(RUN),'zip',RUN)
print(note)
print('导出包：',archive)


阶段核查已完成。缓存：PASS。统一训练入口：CE, KD_raw, KD_calibrated, KD_base, KD_hard, CE_image, CE_text。已完成 GPU 短跑，具体耗时见 timing_summary.csv。本次只核查既有缓存并测量临时分类头训练步，未新增正式模型或测试成绩。
导出包： /content/drive/MyDrive/CLIP_Jev_阶段缓存短跑_20261022/short_run_20261023_155033_088486.zip
